# Sentence token contributions

Token attributions by outcome and by position in the target sentence or its context. The reference-score analyses use the selected binary SHAP table.

In [ ]:
from shared import paths
from IPython.display import display


## Setup

In [ ]:
paths.prepare_directories()


## Global token contributions

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import re
import spacy
from nltk.corpus import stopwords
import nltk

nltk.download('stopwords', quiet=True)
stop_words = set(stopwords.words('english'))
nlp = spacy.load('en_core_web_sm')


def plot_global_shap_summary(
    csv_path,
    output_dir,
    n_pos=15,
    n_neg=10,
    max_token_length=20,
    min_count=3,
    round_zero=2,
):
    excluded_tokens = {
        'pickwick',
        'oliver',
        'twist',
        'nickleby',
        'curiosity',
        'shop',
        'barnaby',
        'rudge',
        'chuzzlewit',
        'dombey',
        'son',
        'david',
        'copperfield',
        'bleak',
        'house',
        'hard',
        'times',
        'little',
        'dorrit',
        'tale',
        'two',
        'cities',
        'great',
        'expectations',
        'mutual',
        'friend',
        'edwin',
        'drood',
        'scrooge',
        'tim',
        'fagin',
        'dodger',
        'uriah',
        'heep',
        'micawber',
        'pip',
        'havisham',
        'estella',
        'sydney',
        'carton',
        'charles',
        'darnay',
        'madame',
        'defarge',
        'samuel',
        'weller',
        'wackford',
        'squeers',
        'bumble',
        'nancy',
        'sikes',
        'tales',
        'dickens',
        'faithfully',
        'esquire',
        'dear',
        'rochester',
        'devonshire',
        'higham',
        'tavistock',
        'letter',
        'wills',
        'hill',
        'coutts',
        'office',
        'leech',
        'carol',
        'affectionately',
        'mr',
    }

    def is_clean_word(token):
        if not isinstance(token, str):
            return False
        token = token.strip()
        if not token or token in {'[CLS]', '[SEP]'}:
            return False
        return re.fullmatch('[A-Za-z]+', token) is not None

    def lemmatize_token(token):
        if not token:
            return token
        doc = nlp(token)
        return doc[0].lemma_
    df = pd.read_csv(csv_path)
    if df.empty:
        print(f" CSV '{csv_path}' is empty.")
        return
    df = df.dropna(subset=['token', 'shap_Ref']).copy()
    df['token'] = df['token'].astype(str)
    df['shap_Ref'] = pd.to_numeric(df['shap_Ref'], errors='coerce')
    df = df.dropna(subset=['shap_Ref'])
    if round_zero is not None:
        df = df[df['shap_Ref'].round(round_zero) != 0]
    if df.empty:
        print(' No SHAP values remain after the zero-value filter.')
        return
    df['token_lower'] = df['token'].str.lower()
    df = df[(df['token'].str.len() <= max_token_length) & df['token_lower'].apply(is_clean_word) & ~df['token_lower'].isin(stop_words)].copy()
    if df.empty:
        print(' No tokens remain after filtering.')
        return
    df['lemma'] = df['token_lower'].apply(lemmatize_token)
    counts = df['lemma'].value_counts()
    valid = set(counts[counts >= min_count].index)
    df = df[df['lemma'].isin(valid)].copy()
    if df.empty:
        print(' No lemmas meet the minimum count.')
        return
    shap_sum = df.groupby('lemma', as_index=False)['shap_Ref'].sum()
    shap_mean = df.groupby('lemma', as_index=False)['shap_Ref'].mean()

    def plot_shap(df_agg, value_col, mode, output_path):
        pos = df_agg[df_agg[value_col] > 0].nlargest(n_pos, value_col)
        neg = df_agg[df_agg[value_col] < 0].nsmallest(n_neg, value_col)
        tokens = list(pos['lemma']) + list(neg['lemma'][::-1])
        pos_vals = list(pos[value_col]) + [0] * len(neg)
        neg_vals = [0] * len(pos) + list(neg[value_col][::-1])
        y = range(len(tokens))
        plt.figure(figsize=(10, 0.6 * len(tokens)))
        sns.set_style('whitegrid')
        fig, ax = plt.subplots(figsize=(10, 0.6 * len(tokens)))
        ax.barh(y, neg_vals, alpha=0.75, label='negative', color='blue')
        ax.barh(y, pos_vals, alpha=0.75, label='positive', color='red')
        ax.axvline(0, color='black', lw=0.8)
        ax.set_yticks(y)
        ax.set_yticklabels(tokens, fontsize=10)
        ax.invert_yaxis()
        ax.set_xlabel('SHAP Value')
        for i, (p, n) in enumerate(zip(pos_vals, neg_vals)):
            if p != 0:
                ax.text(
                    p + 0.01 * abs(p if p != 0 else 1),
                    i,
                    f'{p:+.2f}',
                    va='center',
                    ha='left',
                    fontsize=8,
                )
            if n != 0:
                ax.text(
                    n - 0.01 * abs(n if n != 0 else 1),
                    i,
                    f'{n:+.2f}',
                    va='center',
                    ha='right',
                    fontsize=8,
                )
        plt.tight_layout()
        os.makedirs(output_dir, exist_ok=True)
        plt.savefig(output_path, dpi=300)
        plt.close()
        print(f' Plot saved: {output_path}')
    os.makedirs(output_dir, exist_ok=True)
    plot_shap(
        shap_sum,
        'shap_Ref',
        'Sum',
        os.path.join(output_dir, 'shap_summary_sum_without_exclude.png'),
    )
    plot_shap(
        shap_mean,
        'shap_Ref',
        'Mean',
        os.path.join(output_dir, 'shap_summary_mean_without_exclude.png'),
    )
plot_global_shap_summary(
    csv_path=str(paths.sentence_run(3) / 'general/shap_output_all_folds.csv'),
    output_dir=str(paths.sentence_run(3) / 'general/plots/'),
    n_pos=10,
    n_neg=10,
)


## Classification outcomes

In [ ]:
import re, ast
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import spacy
from nltk.corpus import stopwords
import nltk

nltk.download('stopwords', quiet=True)
STOP_WORDS = set(stopwords.words('english'))
NLP = spacy.load('en_core_web_sm')
CSV_PATH = str(paths.sentence_run(3) / 'general/shap_output_all_folds.csv')
MAX_LEN = 20
MIN_COUNT = 3
AGGREGATE = 'sum'
EXCLUDE = {
    'pickwick',
    'oliver',
    'twist',
    'nickleby',
    'curiosity',
    'shop',
    'barnaby',
    'rudge',
    'chuzzlewit',
    'dombey',
    'son',
    'david',
    'copperfield',
    'bleak',
    'house',
    'hard',
    'times',
    'little',
    'dorrit',
    'tale',
    'two',
    'cities',
    'great',
    'expectations',
    'mutual',
    'friend',
    'edwin',
    'drood',
    'scrooge',
    'tim',
    'fagin',
    'dodger',
    'uriah',
    'heep',
    'micawber',
    'pip',
    'havisham',
    'estella',
    'sydney',
    'carton',
    'charles',
    'darnay',
    'madame',
    'defarge',
    'samuel',
    'weller',
    'wackford',
    'squeers',
    'bumble',
    'nancy',
    'sikes',
    'tales',
    'dickens',
    'faithfully',
    'esquire',
    'dear',
    'rochester',
    'devonshire',
    'higham',
    'tavistock',
    'letter',
    'wills',
    'hill',
    'coutts',
    'office',
    'leech',
    'carol',
    'affectionately',
    'mr',
}


def parse_label_list(x):
    if isinstance(x, (list, tuple, np.ndarray)):
        return [int(v) for v in x]
    if isinstance(x, str):
        try:
            return [int(v) for v in ast.literal_eval(x)]
        except Exception:
            return []
    try:
        return [int(x)]
    except Exception:
        return []


def to_ref_binary(labels):
    arr = parse_label_list(labels)
    return int(any((v == 1 for v in arr))) if arr else 0


def outcome_of_row(true_labels, pred_labels):
    t, p = (to_ref_binary(true_labels), to_ref_binary(pred_labels))
    if t == 1 and p == 1:
        return 'TP'
    if t == 0 and p == 1:
        return 'FP'
    if t == 1 and p == 0:
        return 'FN'
    return 'TN'


def is_clean_word(token, max_len):
    if not isinstance(token, str):
        return False
    tok = token.strip()
    if not tok or tok in {'[CLS]', '[SEP]'}:
        return False
    if len(tok) > max_len:
        return False
    return re.fullmatch('[A-Za-z]+', tok) is not None


def lemmatize_token(token: str) -> str:
    if not token:
        return token
    doc = NLP(token)
    return doc[0].lemma_
df = pd.read_csv(CSV_PATH)
df = df.dropna(subset=['token', 'shap_Ref']).copy()
df['token'] = df['token'].astype(str)
df['shap_Ref'] = pd.to_numeric(df['shap_Ref'], errors='coerce')
df = df.dropna(subset=['shap_Ref'])
df['outcome'] = [outcome_of_row(t, p) for t, p in zip(df['true_labels'], df['pred_labels'])]
df['token_lower'] = df['token'].str.lower()
mask = df['token_lower'].apply(lambda t: is_clean_word(t, MAX_LEN)) & ~df['token_lower'].isin(STOP_WORDS)
df = df[mask].copy()
df['lemma'] = df['token_lower'].apply(lemmatize_token)
vc = df['lemma'].value_counts()
valid = set(vc[vc >= MIN_COUNT].index)
df = df[df['lemma'].isin(valid)].copy()
df_pos = df[df['shap_Ref'] > 0].copy()
if AGGREGATE == 'sum':
    agg = df_pos.groupby(['outcome', 'lemma'])['shap_Ref'].agg(value='sum', count='size').reset_index()
else:
    agg = df_pos.groupby(['outcome', 'lemma'])['shap_Ref'].agg(value='mean', count='size').reset_index()
AGG_TP = agg[agg['outcome'] == 'TP'].drop(columns=['outcome']).sort_values('value', ascending=False)
AGG_FP = agg[agg['outcome'] == 'FP'].drop(columns=['outcome']).sort_values('value', ascending=False)
AGG_FN = agg[agg['outcome'] == 'FN'].drop(columns=['outcome']).sort_values('value', ascending=False)
print(f'Prepared (positive-only): TP={len(AGG_TP)}, FP={len(AGG_FP)}, FN={len(AGG_FN)}')
N_POS = 10
AGG_LABEL = 'Summed SHAP' if AGGREGATE == 'sum' else 'Mean SHAP'
sns.set_style('whitegrid')


def plot_single(agg_df, title):
    sub = agg_df[agg_df['value'] > 0].nlargest(N_POS, 'value')
    if sub.empty:
        print(f' {title}: no positive tokens.')
        return
    y = range(len(sub))
    plt.figure(figsize=(8, 0.5 * len(sub)))
    plt.barh(y, sub['value'])
    plt.yticks(y, sub['lemma'])
    plt.gca().invert_yaxis()
    plt.xlabel(AGG_LABEL)
    plt.title(title)
    plt.tight_layout()
    plt.show()
plot_single(AGG_TP, 'Top tokens (TP) — Ref class')
plot_single(AGG_FP, 'Top tokens (FP) — Ref class')
plot_single(AGG_FN, 'Top tokens (FN) — Ref class')
fig, axes = plt.subplots(1, 3, figsize=(18, 0.5 * N_POS))
for ax, (name, data) in zip(axes, [('TP', AGG_TP), ('FP', AGG_FP), ('FN', AGG_FN)]):
    sub = data[data['value'] > 0].nlargest(N_POS, 'value')
    ax.grid(True, axis='x', alpha=0.2)
    if sub.empty:
        ax.set_title(f'{name} (no positives)')
        ax.set_xticks([])
        ax.set_yticks([])
        continue
    y = range(len(sub))
    ax.barh(y, sub['value'])
    ax.set_yticks(y)
    ax.set_yticklabels(sub['lemma'])
    ax.invert_yaxis()
    ax.set_title(name)
for ax in axes:
    ax.set_xlabel(AGG_LABEL)
plt.suptitle('Top positive tokens per outcome — Ref class')
plt.tight_layout()
plt.show()


## Correct and erroneous classifications

In [ ]:
AGG_ERR = pd.concat([AGG_FP, AGG_FN], ignore_index=True).groupby('lemma', as_index=False).agg(
    value=('value', 'sum'),
    count=('count', 'sum'),
).sort_values(
    'value',
    ascending=False,
)
TP_TOP10 = AGG_TP[AGG_TP['value'] > 0].nlargest(10, 'value').copy()
ERR_TOP10 = AGG_ERR[AGG_ERR['value'] > 0].nlargest(10, 'value').copy()
tp_set = set(TP_TOP10['lemma'])
err_set = set(ERR_TOP10['lemma'])
tp_only = tp_set - err_set
err_only = err_set - tp_set
shared = tp_set & err_set


def plot_exclusives():
    parts = []
    if tp_only:
        parts.append(
            pd.DataFrame(
                {
                    'lemma': [l for l in tp_only],
                    'count': TP_TOP10.set_index('lemma').loc[list(tp_only), 'count'].values,
                    'class': 'TP (exclusive)',
                },
            ),
        )
    if err_only:
        parts.append(
            pd.DataFrame(
                {
                    'lemma': [l for l in err_only],
                    'count': ERR_TOP10.set_index('lemma').loc[list(err_only), 'count'].values,
                    'class': 'ERR (FP∪FN, exclusive)',
                },
            ),
        )
    if not parts:
        print('No exclusive lemmas between TP and ERR Top-10.')
        return
    ex_df = pd.concat(parts, ignore_index=True)
    ex_df = ex_df.sort_values(['class', 'count'], ascending=[True, False])
    plt.figure(figsize=(10, 0.5 * max(4, len(ex_df))))
    sns.barplot(data=ex_df, x='count', y='lemma', hue='class', dodge=False)
    plt.xlabel('Token frequency (count)')
    plt.ylabel('Exclusive Top-10 lemmas')
    plt.title('Exclusives in Top-10: TP vs. ERR (FP∪FN)')
    plt.tight_layout()
    plt.show()
plot_exclusives()


def plot_shared_comparison():
    if not shared:
        print('No shared lemmas between TP and ERR Top-10.')
        return
    tp_counts = TP_TOP10.set_index('lemma').loc[list(shared), 'count']
    err_counts = ERR_TOP10.set_index('lemma').loc[list(shared), 'count']
    cmp_df = pd.DataFrame(
        {'lemma': list(shared), 'TP_count': tp_counts.values, 'ERR_count': err_counts.values},
    )
    m = cmp_df.melt(
        id_vars='lemma',
        value_vars=['TP_count', 'ERR_count'],
        var_name='class',
        value_name='count',
    )
    m['class'] = m['class'].map({'TP_count': 'TP', 'ERR_count': 'ERR (FP∪FN)'})
    order = cmp_df[['TP_count', 'ERR_count']].max(axis=1).rename('maxc').to_frame().assign(
        lemma=cmp_df['lemma'],
    ).sort_values(
        'maxc',
        ascending=False,
    )['lemma'].tolist()
    plt.figure(figsize=(10, 0.55 * max(4, len(shared))))
    sns.barplot(data=m, x='count', y='lemma', hue='class', order=order)
    plt.xlabel('Token frequency (count)')
    plt.ylabel('Shared Top-10 lemmas')
    plt.title('Shared Top-10: which class uses them more often?')
    plt.tight_layout()
    plt.show()
plot_shared_comparison()
TOP_N = 15
AGG_LABEL = 'Summed SHAP' if AGGREGATE == 'sum' else 'Mean SHAP'
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')


def plot_error_tokens(agg_df, title):
    sub = agg_df[agg_df['value'] > 0].nlargest(TOP_N, 'value').copy()
    if sub.empty:
        print(f' {title}: no positive-contribution tokens found.')
        return
    plt.figure(figsize=(8, 0.5 * max(4, len(sub))))
    plt.barh(range(len(sub)), sub['value'])
    plt.yticks(range(len(sub)), sub['lemma'])
    plt.gca().invert_yaxis()
    plt.xlabel(AGG_LABEL)
    plt.title(title)
    plt.tight_layout()
    plt.show()
plot_error_tokens(AGG_FP, 'Top error-driving tokens — FP (aggregated SHAP)')
plot_error_tokens(AGG_FN, 'Top error-driving tokens — FN (aggregated SHAP)')


## Input segments

In [ ]:
import pandas as pd
import ast

CSV_PATH = str(paths.sentence_run(3) / 'general/shap_output_all_folds.csv')


def parse_label_list(x):
    if isinstance(x, str):
        try:
            return [int(v) for v in ast.literal_eval(x)]
        except Exception:
            return []
    if isinstance(x, (list, tuple)):
        return [int(v) for v in x]
    return []
df = pd.read_csv(CSV_PATH)
df = df.dropna(subset=['shap_Ref', 'sent_id', 'token']).copy()
df['shap_Ref'] = pd.to_numeric(df['shap_Ref'], errors='coerce')
df = df.dropna(subset=['shap_Ref'])
df['outcome'] = [outcome_of_row(t, p) for t, p in zip(df['true_labels'], df['pred_labels'])]
df = df[df['outcome'].isin(['TP', 'FP', 'FN'])].copy()
df['group'] = df['outcome'].apply(lambda x: 'Correct' if x == 'TP' else 'Error')


def mark_target_zone(group):
    in_target = False
    segment_types = []
    for tok in group['token']:
        if '<target' in tok:
            in_target = True
            segment_types.append('marker')
        elif '</target' in tok:
            in_target = False
            segment_types.append('marker')
        else:
            segment_types.append('target' if in_target else 'context')
    group['segment_type'] = segment_types
    return group
df = df.groupby('sent_id', group_keys=False).apply(mark_target_zone)
df = df[df['segment_type'].isin({'target', 'context'})].copy()
sum_target = df.loc[df['segment_type'] == 'target', 'shap_Ref'].sum()
sum_context = df.loc[df['segment_type'] == 'context', 'shap_Ref'].sum()
len_target = (df['segment_type'] == 'target').sum()
len_context = (df['segment_type'] == 'context').sum()
avg_target = sum_target / len_target
avg_context = sum_context / len_context
print('=== Global Summary ===')
print(f'∑ SHAP target:  {sum_target:.4f}  (n = {len_target})')
print(f'∑ SHAP context: {sum_context:.4f}  (n = {len_context})')
print(f'Avg per token – target:  {avg_target:.6f}')
print(f'Avg per token – context: {avg_context:.6f}\n')
results = []
for group_name in ['Correct', 'Error']:
    group_df = df[df['group'] == group_name]
    for segment in ['target', 'context']:
        subset = group_df[group_df['segment_type'] == segment]
        sum_shap = subset['shap_Ref'].sum()
        n_tokens = len(subset)
        avg_shap = sum_shap / n_tokens if n_tokens > 0 else 0
        results.append(
            {
                'Group': group_name,
                'Segment': segment,
                'Sum SHAP': round(sum_shap, 2),
                'Token Count': n_tokens,
                'Avg SHAP per Token': round(avg_shap, 6),
            },
        )
summary_df = pd.DataFrame(results)
print('=== Split Summary (Correct vs. Error) ===')
print(summary_df)


## Target and context vocabulary

In [ ]:
import pandas as pd
import spacy
from nltk.corpus import stopwords
import nltk

nltk.download('stopwords')
stop_words = set(stopwords.words('english'))
nlp = spacy.load('en_core_web_sm')
CSV_PATH = str(paths.sentence_run(3) / 'general/shap_output_all_folds.csv')
df = pd.read_csv(CSV_PATH)
df = df.dropna(subset=['shap_Ref', 'sent_id', 'token']).copy()
df['shap_Ref'] = pd.to_numeric(df['shap_Ref'], errors='coerce')
df = df.dropna(subset=['shap_Ref'])


def mark_target_zone(group):
    in_target = False
    segment_types = []
    for tok in group['token']:
        if tok == '<target>':
            in_target = True
            segment_types.append('marker')
        elif tok == '</target>':
            in_target = False
            segment_types.append('marker')
        else:
            segment_types.append('target' if in_target else 'context')
    group['segment_type'] = segment_types
    return group
df = df.groupby('sent_id', group_keys=False).apply(mark_target_zone)
df = df[df['segment_type'].isin({'target', 'context'})].copy()


def lemmatise(tok):
    doc = nlp(tok.lower())
    return doc[0].lemma_ if doc else tok.lower()
df['lemma'] = df['token'].apply(lemmatise)
df = df[df['lemma'].str.isalpha()]
df = df[~df['lemma'].isin(stop_words)]
agg = df.groupby(['lemma', 'segment_type'])['shap_Ref'].sum().reset_index()
pivot = agg.pivot(index='lemma', columns='segment_type', values='shap_Ref').fillna(0.0)
pivot['total'] = pivot['target'] + pivot['context']
pivot = pivot.sort_values('total', ascending=False)
top_total = pivot.sort_values('total', ascending=False).head(15).reset_index()
table_total = top_total[['lemma', 'target', 'context', 'total']]
display(table_total)
top_target = pivot.sort_values('target', ascending=False).head(15).reset_index()
table_target = top_target[['lemma', 'target', 'context', 'total']]
display(table_target)
top_context = pivot.sort_values('context', ascending=False).head(15).reset_index()
table_context = top_context[['lemma', 'target', 'context', 'total']]
display(table_context)


## Target-context differences

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

OUTPUT_DIR = str(paths.FIGURES_DIR)
FILENAME_TARGET = 'shap_diffs_target_gt_context.png'
FILENAME_CONTEXT = 'shap_diffs_context_gt_target.png'
filtered = pivot[~((pivot['target'] < 0) & (pivot['context'] < 0))].copy()
filtered['diff_target_minus_context'] = filtered['target'] - filtered['context']
filtered['diff_context_minus_target'] = -filtered['diff_target_minus_context']
top_target_heavy = filtered[filtered['diff_target_minus_context'] > 0]
top_target_heavy = top_target_heavy.sort_values('diff_target_minus_context', ascending=False).head(15)
table_target_heavy = top_target_heavy[['diff_target_minus_context']].rename(
    columns={'diff_target_minus_context': 'Δ SHAP Value'},
)
top_context_heavy = filtered[filtered['diff_context_minus_target'] > 0]
top_context_heavy = top_context_heavy.sort_values('diff_context_minus_target', ascending=False).head(15)
table_context_heavy = top_context_heavy[['diff_context_minus_target']].rename(
    columns={'diff_context_minus_target': 'Δ SHAP Value'},
)
FILENAME_TARGET = 'shap_diffs_target_gt_context.png'
FILENAME_CONTEXT = 'shap_diffs_context_gt_target.png'
EXPORT_DIR = str(paths.FIGURES_DIR)
vmin = 0
vmax = 12
cmap = 'Reds'
annot_size = 18
figsize = (5, 6)
blacklist = {'I', 'dicken'}
filtered_target = table_target_heavy[~table_target_heavy.index.isin(blacklist)].copy()
filtered_context = table_context_heavy[~table_context_heavy.index.isin(blacklist)].copy()


def clean_axis(ax):
    ax.set_xlabel('')
    ax.set_ylabel('')
fig, ax = plt.subplots(figsize=figsize)
sns.heatmap(
    filtered_target,
    annot=True,
    fmt='.2f',
    annot_kws={'size': annot_size},
    cmap=cmap,
    cbar=False,
    linewidths=0.5,
    vmin=vmin,
    vmax=vmax,
)
clean_axis(ax)
ax.tick_params(axis='y', labelsize=18)
plt.tight_layout()
plt.savefig(os.path.join(EXPORT_DIR, FILENAME_TARGET), dpi=300)
plt.show()
fig, ax = plt.subplots(figsize=figsize)
sns.heatmap(
    filtered_context,
    annot=True,
    fmt='.2f',
    annot_kws={'size': annot_size},
    cmap=cmap,
    cbar=True,
    cbar_kws={'label': '', 'shrink': 1.0},
    linewidths=0.5,
    vmin=vmin,
    vmax=vmax,
)
clean_axis(ax)
ax.tick_params(axis='y', labelsize=18)
plt.tight_layout()
plt.savefig(os.path.join(EXPORT_DIR, FILENAME_CONTEXT), dpi=300)
plt.show()
